# Huggingface pipeline

Install all required packages for the fine-tuning experiment.

In [1]:
import importlib.metadata
import subprocess
import sys

def install_dep(modules: list):
    for pkg in modules:
        base = pkg.split("[")[0]  # strip extras like [torch]
        try:
            importlib.metadata.distribution(base)
            print(f"{pkg} already installed")
        except importlib.metadata.PackageNotFoundError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

modules = ["huggingface_hub",
           "datasets", 
           "transformers", 
           "torch", 
           "numpy", 
           "nvidia-ml-py", 
           "transformers[torch]", 
           "evaluate",
           "ipywidgets",
           "scikit-learn",
           "scipy",
           "accelerate"]
install_dep(modules)

huggingface_hub already installed
datasets already installed
transformers already installed
torch already installed
numpy already installed
nvidia-ml-py already installed
transformers[torch] already installed
evaluate already installed
ipywidgets already installed
scikit-learn already installed
scipy already installed
accelerate already installed


Login with notebook to Huggingface

In [2]:
from huggingface_hub import notebook_login
notebook_login()

Load the MRPC dataset from GLUE — 3 668 training and 408 validation sentence pairs labeled as paraphrase or not.

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

Load the BERT tokenizer, define a tokenisation function for sentence pairs, and create a `DataCollatorWithPadding` for dynamic batching.

In [5]:
from transformers import AutoTokenizer, DataCollatorWithPadding
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_function(item):
    return tokenizer(item["sentence1"],item["sentence2"],truncation=True,padding=True,return_tensors="pt")
data_collactor = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Apply tokenisation to train and validation sets; drop raw text columns, rename `label` → `labels`, and format as PyTorch tensors.

In [6]:
train_dataset = raw_datasets["train"]
train_dataset = train_dataset.map(tokenizer_function,batched=True)
train_dataset = train_dataset.remove_columns(["sentence1","sentence2","idx"])
train_dataset = train_dataset.rename_column("label","labels")
train_dataset.set_format("torch")

validation_dataset = raw_datasets["validation"]
validation_dataset = validation_dataset.map(tokenizer_function,batched=True)
validation_dataset = validation_dataset.remove_columns(["sentence1","sentence2","idx"])
validation_dataset = validation_dataset.rename_column("label","labels")
validation_dataset.set_format("torch")


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Load pretrained `bert-base-uncased` with a fresh 2-class classification head (the MLM/NSP head weights are discarded).

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TODO: update

In [60]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader (train_dataset, shuffle=True, batch_size=8,collate_fn=data_collactor)
validation_dataloader = DataLoader (validation_dataset, batch_size=8,collate_fn=data_collactor)

TODO: update

In [64]:
import torch
for batch in validation_dataloader:
    break
with torch.no_grad():
    prediction = model.forward(**batch)
prediction

SequenceClassifierOutput(loss=tensor(0.7177), logits=tensor([[0.6830, 0.2503],
        [0.6603, 0.2457],
        [0.6725, 0.2640],
        [0.6817, 0.2679],
        [0.6845, 0.2587],
        [0.6569, 0.2418],
        [0.6664, 0.2630],
        [0.6924, 0.2573]]), hidden_states=None, attentions=None)

TODO: update

In [65]:
# Load the optimizer
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),lr=5e-5)

In [67]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377


TODO: update

In [68]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device
model.to(device)

device(type='cpu')

In [71]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

Finish
Finish
Finish


TODO: update

In [73]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in validation_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.6838235294117647, 'f1': 0.8122270742358079}